# Plots for First Hit Analysis

The plots created in this notebook aim to investigate the use of the first hit in a track to initialise the dynamic queries.

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from track_evaluate import load_events

In [ ]:
training_colours = {
    "first_hit": "tab:orange",
    "last_hit": "tab:blue"
}

tracking_fnames = {
    "first_hit": "/share/rcif2/mmangat/logs/TRK-v8-3l-fast-fix-vec_20260316-T104827/ckpts/epoch=020-val_loss=1.48183_test_eval.h5",
    "last_hit": "/share/rcif2/mmangat/logs/TRK-v8-3l-fast-fix-vec_20260428-T172805/ckpts/epoch=004-val_loss=0.58669_test_eval.h5",
}

tracking_input = "/share/rcif2/pduckett/data/prepped/test/"

In [ ]:
def load_vertex_positions(h5_path, data_dir):
    """Load vx, vy, vz for each particle slot by joining H5 particle_ids with parquet files.

    Returns a DataFrame indexed the same way as parts from load_events (event_id, particle slot order).
    """
    rows = []
    with h5py.File(h5_path, "r") as f:
        for event_id in f.keys():
            particle_ids = np.array(f[f"{event_id}/targets/particle_id"][0])  # (N_slots,)

            parquet_path = Path(data_dir) / f"event{event_id}-parts.parquet"
            parts_df = pd.read_parquet(parquet_path, columns=["particle_id", "vx", "vy", "vz"])
            id_to_row = parts_df.set_index("particle_id")

            for pid in particle_ids:
                if pid in id_to_row.index:
                    row = id_to_row.loc[pid]
                    rows.append({"event_id": event_id, "vx": row["vx"], "vy": row["vy"], "vz": row["vz"]})
                else:
                    rows.append({"event_id": event_id, "vx": np.nan, "vy": np.nan, "vz": np.nan})

    return pd.DataFrame(rows)


# Load vertex positions for each model's H5 file
import h5py
from pathlib import Path

vertex_data = {
    name: load_vertex_positions(fname, tracking_input)
    for name, fname in tracking_fnames.items()
}

In [ ]:
def load_hits_for_nan_vertex_particles(h5_path, data_dir, coord_fields=None):
    """For each event, find particles with NaN vx/vy/vz and return the hit coordinates for those particles."""
    if coord_fields is None:
        coord_fields = ["x", "y", "z", "r", "eta", "phi"]

    rows = []
    with h5py.File(h5_path, "r") as f:
        for event_id in f.keys():
            particle_ids = np.array(f[f"{event_id}/targets/particle_id"][0])       # (N_slots,)
            hit_particle_ids = np.array(f[f"{event_id}/targets/hit_particle_id"][0])  # (N_hits,)

            parquet_path = Path(data_dir) / f"event{event_id}-parts.parquet"
            parts_df = pd.read_parquet(parquet_path, columns=["particle_id", "vx", "vy", "vz"])
            id_to_vertex = parts_df.set_index("particle_id")

            # Look up vertex coordinates for every particle slot
            def get_vertex(pid, col):
                return id_to_vertex.loc[pid, col] if pid in id_to_vertex.index else np.nan

            vx = np.array([get_vertex(pid, "vx") for pid in particle_ids])
            vy = np.array([get_vertex(pid, "vy") for pid in particle_ids])
            vz = np.array([get_vertex(pid, "vz") for pid in particle_ids])

            # Real particles (not padding) whose vertex is NaN in at least one coordinate
            nan_mask = (np.isnan(vx) | np.isnan(vy) | np.isnan(vz)) & (particle_ids != -999)
            nan_vertex_pids = particle_ids[nan_mask]

            if len(nan_vertex_pids) == 0:
                continue

            # Find hits that belong to any of those particles
            hit_mask = np.isin(hit_particle_ids, nan_vertex_pids)
            if not hit_mask.any():
                continue

            # Load hit coordinates from the H5 inputs
            hit_coords = {}
            for field in coord_fields:
                key = f"{event_id}/inputs/hit_{field}"
                if key in f:
                    hit_coords[field] = np.array(f[key][0])[hit_mask]

            hit_pids = hit_particle_ids[hit_mask]
            for i in range(hit_mask.sum()):
                row = {"event_id": int(event_id), "particle_id": int(hit_pids[i])}
                for field in coord_fields:
                    if field in hit_coords:
                        row[f"hit_{field}"] = float(hit_coords[field][i])
                rows.append(row)

    return pd.DataFrame(rows)


nan_vertex_hits = {
    name: load_hits_for_nan_vertex_particles(fname, tracking_input)
    for name, fname in tracking_fnames.items()
}

# Save to parquet alongside the H5 files
for name, fname in tracking_fnames.items():
    out_path = Path(fname).with_suffix("") .parent / f"nan_vertex_hits_{name}.parquet"
    nan_vertex_hits[name].to_parquet(out_path, index=False)
    print(f"{name}: {len(nan_vertex_hits[name])} hits saved to {out_path}")